## SAT179_4_iNaturalist

* [#425](https://github.com/salgo60/Stockholm_Archipelago_Trail/issues/425) "ss"
* this notebook [425_Interactive_Photo_Gallery.ipynb](https://github.com/salgo60/Stockholm_Archipelago_Trail/tree/main/Notebook/425_Interactive_Photo_Gallery.ipynb) 


In [1]:
import time
import datetime  
start_time = time.time()
start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Started: {start_str}")


Started: 2026-06-24 05:29


In [7]:

from pathlib import Path
import json
import time

import pandas as pd
import requests
import folium

from shapely.geometry import (
    shape,
    LineString,
    MultiLineString,
    Point,
)

from shapely.ops import unary_union

# =====================================================
# CONFIG
# =====================================================

TRAIL_FILE = Path("SAT_full.geojson")

CACHE_FILE = Path(
    "inat_observations.csv"
)

OUTPUT_HTML = Path(
    "SAT_iNaturalist_500m.html"
)

BUFFER_DEG = 0.005

INAT_URL = (
    "https://api.inaturalist.org/v1/observations"
)

TAXA_GROUPS = [

    ("Aves", "Birds"),
    ("Plantae", "Plants"),
    ("Mammalia", "Mammals"),
    ("Fungi", "Fungi"),
    ("Insecta", "Insects"),
]

# =====================================================
# LOAD SAT
# =====================================================

print("Loading SAT trail")

with TRAIL_FILE.open(
    encoding="utf-8"
) as f:

    gj = json.load(f)

lines = []

for feat in gj["features"]:

    geom = shape(
        feat["geometry"]
    )

    if isinstance(
        geom,
        LineString
    ):

        lines.append(geom)

    elif isinstance(
        geom,
        MultiLineString
    ):

        lines.extend(
            list(geom.geoms)
        )

trail = unary_union(lines)

print(
    f"Loaded {len(lines)} segments"
)

# =====================================================
# 500 M BUFFER
# =====================================================

search_geom = trail.buffer(
    BUFFER_DEG
)

minx, miny, maxx, maxy = (
    search_geom.bounds
)

print(
    "Search bbox:"
)

print(
    minx,
    miny,
    maxx,
    maxy
)

# =====================================================
# DOWNLOAD FROM INATURALIST
# =====================================================

def get_display_name(obs):

    taxon = (
        obs.get("taxon")
        or {}
    )

    return (
        taxon.get(
            "preferred_common_name"
        )
        or taxon.get("name")
        or obs.get(
            "species_guess"
        )
        or "Unknown"
    )

def fetch_taxon(
    iconic_taxa
):

    print(
        f"Downloading {iconic_taxa}"
    )

    records = []

    page = 1

    while True:

        params = {

            "iconic_taxa":
                iconic_taxa,

            "swlat":
                miny,

            "swlng":
                minx,

            "nelat":
                maxy,

            "nelng":
                maxx,

            "per_page":
                200,

            "page":
                page,

            "order":
                "desc",

            "order_by":
                "observed_on",
        }

        r = requests.get(
            INAT_URL,
            params=params,
            timeout=30,
        )

        r.raise_for_status()

        data = r.json()

        results = data.get(
            "results",
            []
        )

        if not results:
            break

        for obs in results:

            gj = (
                obs.get("geojson")
                or {}
            )

            coords = gj.get(
                "coordinates"
            )

            if not coords:
                continue

            lon, lat = coords[:2]

            photos = (
                obs.get("photos")
                or []
            )

            photo_url = ""

            if photos:

                photo_url = (
                    photos[0]
                    .get("url", "")
                    .replace(
                        "square",
                        "medium"
                    )
                )

            taxon = (
                obs.get("taxon")
                or {}
            )

            records.append(
                {

                    "id":
                        obs.get("id"),

                    "species_guess":
                        get_display_name(obs),

                    "scientific_name":
                        taxon.get("name"),

                    "observed_on":
                        obs.get(
                            "observed_on"
                        ),

                    "latitude":
                        lat,

                    "longitude":
                        lon,

                    "photo_url":
                        photo_url,

                    "iconic_taxa":
                        iconic_taxa,
                }
            )

        print(
            iconic_taxa,
            page,
            len(results)
        )

        page += 1

        time.sleep(0.2)

        if page > 20:
            break

    return records

# =====================================================
# CACHE
# =====================================================

if CACHE_FILE.exists():

    print(
        "Using cache"
    )

    df = pd.read_csv(
        CACHE_FILE
    )

else:

    all_records = []

    for taxon, _ in TAXA_GROUPS:

        all_records.extend(
            fetch_taxon(
                taxon
            )
        )

    df = pd.DataFrame(
        all_records
    )

    df.to_csv(
        CACHE_FILE,
        index=False
    )

    print(
        f"Saved {CACHE_FILE}"
    )

print(df.shape)

# =====================================================
# FILTER 500 M
# =====================================================

def near_sat(row):

    pt = Point(
        row["longitude"],
        row["latitude"]
    )

    return (
        search_geom.contains(pt)
        or
        search_geom.touches(pt)
    )

df["near_sat"] = df.apply(
    near_sat,
    axis=1
)

sat_df = df[
    df["near_sat"]
].copy()

print(
    f"{len(sat_df)} observations near SAT"
)

# =====================================================
# MAP
# =====================================================

center = trail.centroid

m = folium.Map(
    location=[
        center.y,
        center.x
    ],
    zoom_start=9,
    control_scale=True,
)

# =====================================================
# DRAW TRAIL
# =====================================================

def iter_lines(g):

    if isinstance(
        g,
        LineString
    ):
        yield g

    elif isinstance(
        g,
        MultiLineString
    ):
        for seg in g.geoms:
            yield seg

for seg in iter_lines(
    trail
):

    coords = [

        (lat, lon)

        for lon, lat
        in seg.coords

    ]

    folium.PolyLine(
        coords,
        color="blue",
        weight=4,
    ).add_to(m)

# =====================================================
# OBSERVATIONS
# =====================================================

for _, row in sat_df.iterrows():

    popup = f"""
    <b>{row['species_guess']}</b><br>
    {row['observed_on']}<br>
    """

    if (
        isinstance(
            row["photo_url"],
            str
        )
        and
        row["photo_url"]
    ):

        popup += f"""
        <img
            src="{row['photo_url']}"
            width="250">
        """

    folium.CircleMarker(

        location=[
            row["latitude"],
            row["longitude"]
        ],

        radius=5,

        color="green",

        fill=True,

        popup=folium.Popup(
            popup,
            max_width=300
        ),

    ).add_to(m)

# =====================================================
# SAVE
# =====================================================

m.save(
    OUTPUT_HTML
)

print()

print(
    "Saved:"
)

print(
    OUTPUT_HTML
)


Loading SAT trail
Loaded 20 segments
Search bbox:
17.8531556242 58.7326640765 19.1441659923 59.8675446646
Aves 1 200
Aves 2 200
Aves 3 200
Aves 4 200
Aves 5 200
Aves 6 200
Aves 7 200
Aves 8 200
Aves 9 200
Aves 10 200
Aves 11 200
Aves 12 200
Aves 13 200
Aves 14 200
Aves 15 200
Aves 16 200
Aves 17 200
Aves 18 200
Aves 19 200
Aves 20 200
Plantae 1 200
Plantae 2 200
Plantae 3 200
Plantae 4 200
Plantae 5 200
Plantae 6 200
Plantae 7 200
Plantae 8 200
Plantae 9 200
Plantae 10 200
Plantae 11 200
Plantae 12 200
Plantae 13 200
Plantae 14 200
Plantae 15 200
Plantae 16 200
Plantae 17 200
Plantae 18 200
Plantae 19 200
Plantae 20 200
Mammalia 1 200
Mammalia 2 200
Mammalia 3 200
Mammalia 4 200
Mammalia 5 200
Mammalia 6 200
Mammalia 7 200
Mammalia 8 200
Mammalia 9 200
Mammalia 10 200
Mammalia 11 200
Mammalia 12 200
Mammalia 13 134
Fungi 1 200
Fungi 2 200
Fungi 3 200
Fungi 4 200
Fungi 5 200
Fungi 6 200
Fungi 7 200
Fungi 8 200
Fungi 9 200
Fungi 10 200
Fungi 11 200
Fungi 12 200
Fungi 13 200
Fungi 14 200


In [8]:
end_time = time.time()
duration = end_time - start_time
print(f"Started: {start_str}")
print(f"Fini§shed in {duration:.2f} seconds.")


Started: 2026-06-24 05:29
Fini§shed in 1823.68 seconds.
